In [1]:
# Import required libraries  
import os  
import json  
import openai  
import pandas as pd    
from dotenv import load_dotenv  
from tenacity import retry, wait_random_exponential, stop_after_attempt  
from azure.core.credentials import AzureKeyCredential  
from azure.search.documents import SearchClient  
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.models import Vector  
from azure.search.documents.indexes.models import (  
    SearchIndex,  
    SearchField,  
    SearchFieldDataType,  
    SimpleField,  
    SearchableField,  
    SearchIndex,  
    SemanticConfiguration,  
    PrioritizedFields,  
    SemanticField,  
    SearchField,  
    SemanticSettings,  
    VectorSearch,  
    VectorSearchAlgorithmConfiguration
)  

In [2]:
# Configure environment variables  
load_dotenv()  
service_endpoint = "https://ztn-copilot-search.search.windows.net"  
index_name = "app-crg-rag-constraint"  
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")  
openai.api_type = 'azure'  
openai.api_key = os.getenv("OPENAI_API_KEY")  
openai.api_base = os.getenv("OPENAI_API_BASE")  
openai.api_version = "2023-05-15"
credential = AzureKeyCredential(key)

In [3]:
def extract_constraints(results):
    '''
    Iterates over results iterator and checks if each item contains 'constraint'. 
    If it does, it appends the constraint to the constraints_list. 
    Finally, it joins all constraints into a single string.
    '''
    constraints_list = []  
    for result in results:  
        if 'constraint' in result:  
            constraints_list.append(result['constraint'])  
      
    # join all constraints into a single string  
    constraints_string = ' '.join(constraints_list)  
    return constraints_string 

In [4]:
# Generate Document Embeddings using OpenAI Ada 002
# input_data = pd.read_json(path_or_buf='data/labeled_constraint.jsonl', lines=True)

# Read the text-sample.json
with open('../data/rag_constraints.json', 'r', encoding='utf-8') as file:
    input_data = json.load(file)

@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def generate_embeddings(text):
    response = openai.Embedding.create(
        input=text, engine="text-embedding-ada-002")
    embeddings = response['data'][0]['embedding']
    return embeddings


# Generate embeddings for title and content fields
for item in input_data:
    print(item)
    label = item['label']
    constraint = item['constraint']
    label_embeddings = generate_embeddings(label)
    constraint_embeddings = generate_embeddings(constraint)
    item['labelVector'] = label_embeddings
    item['constraintVector'] = constraint_embeddings
    item['@search.action'] = 'upload'

# Output embeddings to docVectors.json file
with open("output/constraintVectors.json", "w") as f:
    json.dump(input_data, f)

{'id': '1', 'label': 'graph, type', 'constraint': 'The data is represented as a networkx graph made up of cloud resource graph in Azure network.'}
{'id': '2', 'label': 'node, type', 'constraint': "node 'type' can be 'virtualmachines', 'Networkinterfaces', 'virtualnetworks', 'networksecuritygroups'. "}
{'id': '3', 'label': 'node, name', 'constraint': "node 'name' is a string depending on its 'type'. "}
{'id': '4', 'label': 'node, properties, virtualmachines', 'constraint': "when 'type'=='virtualmachines', 'properties' has 'osType', 'Networkinterfaces'. when 'type'=='Networkinterfaces', 'properties' has 'virtualnetworks', 'Networkinterfaces'. when 'type'=='virtualnetworks', 'properties' has 'provisioningState', 'addressPrefixes', 'port'. when 'type'=='networksecuritygroups', 'properties' has 'protocol', 'addressPrefixes', 'port', 'priority'."}
{'id': '5', 'label': 'node, properties, Networkinterfaces', 'constraint': "when 'type'=='Networkinterfaces', 'properties' has 'virtualnetworks', '

In [5]:
# Create a search index
index_client = SearchIndexClient(
    endpoint=service_endpoint, credential=credential)
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="label", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True),
    SearchableField(name="constraint", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True),
    SearchField(name="labelVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, dimensions=1536, vector_search_configuration="my-vector-config"),
    SearchField(name="constraintVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, dimensions=1536, vector_search_configuration="my-vector-config"),
]

vector_search = VectorSearch(
    algorithm_configurations=[
        VectorSearchAlgorithmConfiguration(
            name="my-vector-config",
            kind="hnsw",
            hnsw_parameters={
                "m": 4,
                "efConstruction": 400,
                "efSearch": 1000,
                "metric": "cosine"
            }
        )
    ]
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=PrioritizedFields(
        prioritized_keywords_fields=[SemanticField(field_name="label")],
        prioritized_content_fields=[SemanticField(field_name="constraint")]
    )
)

# Create the semantic settings with the configuration
semantic_settings = SemanticSettings(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=index_name, fields=fields,
                    vector_search=vector_search, semantic_settings=semantic_settings)
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 app-crg-rag-constraint created


In [6]:
# Upload some documents to the index
with open('output/constraintVectors.json', 'r') as file:  
    documents = json.load(file)  
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents)  
print(f"Uploaded {len(documents)} documents")

Uploaded 15 documents


In [13]:
# Pure Vector Search
query = "Add a new packet_switch 'ju1.a1.m1.s4c7' on jupiter 1, aggregation block 1, domain 1, with 5 ports, each port has physical_capacity_bps as 1000. Return the new graph."  
  
search_client = SearchClient(service_endpoint, index_name, AzureKeyCredential(key))  
  
results = search_client.search(  
    search_text='',  
    vector=Vector(value=generate_embeddings(query), k=13, fields="constraintVector"),
    select=[ "label", "constraint"] 
)  

for result in results:  
    print(f"Score: {result['@search.score']}")
    print(f"Label: {result['label']}")  
    print(f"Constraint: {result['constraint']}\n")  

Score: 0.85034436
Label: PORT, attribute
Constraint: Each PORT node has an attribute 'physical_capacity_bps'

Score: 0.84235334
Label: capacity
Constraint: When calculating capacity of a node, you need to sum the physical_capacity_bps on the PORT of each hierarchy contains in this node.

Score: 0.840872
Label: PORT, name
Constraint: For example, a PORT node name is ju1.a1.m1.s2c1.p3

Score: 0.8398656
Label: node, hierarchy
Constraint: Hierarchy: CHASSIS contains PACKET_SWITCH, JUPITER contains SUPERBLOCK, SUPERBLOCK contains AGG_BLOCK, AGG_BLOCK contains PACKET_SWITCH, PACKET_SWITCH contains PORT

Score: 0.82618815
Label: node, add
Constraint: Adding new nodes need to consider all hierarchy. For example, adding a new switch requires adding it to the corresponding jupiter, aggregation block, and domain. The name to add on each layer can be inferred from new node name string.

Score: 0.7974908
Label: graph, add
Constraint: When creating a new graph, need to filter nodes and edges with at

In [9]:
results = search_client.search(
    search_text=query,
    vector=Vector(value=generate_embeddings(
        query), k=12, fields="constraintVector"),
    select=["label", "constraint"],
    query_type="semantic", 
    query_language="en-us", 
    semantic_configuration_name='my-semantic-config', 
    query_caption="extractive", 
    query_answer="extractive",
    top=6
)

semantic_answers = results.get_answers()
for answer in semantic_answers:
    if answer.highlights:
        print(f"Semantic Answer: {answer.highlights}")
    else:
        print(f"Semantic Answer: {answer.text}")
    print(f"Semantic Answer Score: {answer.score}\n")

for result in results:
    print(f"label: {result['label']}")
    print(f"constraint: {result['constraint']}")

label: node, add
constraint: Adding new nodes need to consider all hierarchy. For example, adding a new switch requires adding it to the corresponding jupiter, aggregation block, and domain. The name to add on each layer can be inferred from new node name string.
label: PORT, name
constraint: For example, a PORT node name is ju1.a1.m1.s2c1.p3
label: node, type
constraint: 
Each node has a 'type' attribute, in the format of EK_{TYPE}. Note, 'type' must be a list, can include ['EK_SUPERBLOCK', 'EK_CHASSIS', 'EK_RACK', 'EK_AGG_BLOCK', 'EK_JUPITER', 'EK_PORT', 'EK_SPINEBLOCK', 'EK_PACKET_SWITCH', 'EK_CONTROL_POINT', 'EK_CONTROL_DOMAIN'].
label: node, hierarchy
constraint: Hierarchy: CHASSIS contains PACKET_SWITCH, JUPITER contains SUPERBLOCK, SUPERBLOCK contains AGG_BLOCK, AGG_BLOCK contains PACKET_SWITCH, PACKET_SWITCH contains PORT
label: edge, add
constraint: When adding new nodes, you should also add edges based on their relationship with existing nodes.
label: node, attribute
constrai

In [14]:
extract_constraints(results)

''